In [1]:
import pandas as pd
import json
from google import genai
from dotenv import load_dotenv
import os

In [2]:
api=os.getenv("GOOGLE_API_KEY")
gemini_client = genai.Client(api_key=api)


In [7]:
def ask_llms(
    prompts,
    system_prompt="You are a helpful AI Assistant",
    temperature=0.7,
    model_id="gemini-2.5-flash"
):
    final_prompt = f"{system_prompt}\n\nUser: {prompts}"

    response = gemini_client.models.generate_content(
        model=model_id,
        contents=final_prompt,
        config={"temperature": temperature}
    )

    return response.text

In [8]:
ask_llms("explain the tenses in 3-4 lines")

'Tenses are verb forms that show **when** an action or state takes place. They primarily categorize events into three main time frames: **Past** (it happened), **Present** (it is happening now or regularly), and **Future** (it will happen). Additionally, tenses use "aspects" (simple, continuous, perfect, perfect continuous) to further clarify *how* the action unfolds within that time, such as if it\'s ongoing, completed, or habitual.'

In [9]:
#zero shot prompting 
prompts = "Classify the sentiments of the sentence : This movie is very good"

ask_llms(prompts)

'The sentiment of the sentence "This movie is very good" is **Positive**.'

In [10]:
# Few shot  prompt contains clear examples of the expected input and output format
prompt = """
Review: 'Amazing customer service and fast response.'
Sentiment: -> Positive

Review: 'The product stopped working after one day.'
Sentiment: -> Negative

Review: 'The UI is clean and easy to use.'
Sentiment: 
"""

# Sending the engineered prompt to the LLM
response = ask_llms(prompt)
print(response)

-> Positive


In [11]:
# Chain of thought prompt : Providing an example with a step-by-step breakdown forces the model to think sequentially
prompt = """
Question: A clothing store had 50 shirts. They sold 15 shirts on Monday, restocked 10 shirts on Tuesday, and sold 5 more on Wednesday. How many shirts do they have left?
Answer: Let's break it down step-by-step:
1. Initially, the store had 50 shirts.
2. It sold 15 shirts, so the inventory reduced to 50 - 15 = 35 shirts.
3. The store restocked 10 shirts, so we add these: 35 + 10 = 45 shirts.
4. It sold 5 more shirts, so 45 - 5 = 40 shirts.
Therefore, the final inventory is 40 shirts.

Question: A bookstore has 120 books. They sell 35 books in the morning, restock 20 in the afternoon. How many books do they have now?
Answer: Let's break it down step-by-step:
"""

# Sending the prompt to the LLM
response = ask_llms(prompt)
print(response)

1. Initially, the bookstore had 120 books.
2. They sold 35 books, so the inventory reduced to 120 - 35 = 85 books.
3. The store restocked 20 books, so we add these: 85 + 20 = 105 books.
Therefore, the bookstore has 105 books now.


In [13]:
Prompts = """
Question: Janet has 3 boxes of ornaments. Each box contains 12 ornaments. She decorates the tree and uses 24 ornaments. Then, her dog knocks over one of the remaining boxes, breaking 4 ornaments. How many unbroken ornaments does Janet have left?

Answer: Let's think step-by-step.
"""
responses = []
for i in range(3):
    responses.append(ask_llms(Prompts))

print(responses[0])

Let's think step-by-step.

1.  **Calculate the total number of ornaments Janet has initially:**
    *   Janet has 3 boxes.
    *   Each box contains 12 ornaments.
    *   Total ornaments = 3 boxes * 12 ornaments/box = 36 ornaments.

2.  **Calculate the number of ornaments remaining after decorating the tree:**
    *   She started with 36 ornaments.
    *   She uses 24 ornaments.
    *   Ornaments remaining = 36 - 24 = 12 ornaments.

3.  **Determine which box was knocked over and how many ornaments were in it:**
    *   She used 24 ornaments. Since each box has 12, she used the contents of 2 full boxes (24 / 12 = 2).
    *   This means one full box, containing 12 ornaments, was left untouched. This is the "one of the remaining boxes" her dog knocked over.

4.  **Calculate the number of unbroken ornaments after the dog incident:**
    *   The remaining 12 ornaments were in the box that was knocked over.
    *   4 ornaments broke from that box.
    *   Unbroken ornaments left = 12 - 4 = 8

In [15]:
print(responses[1])

Let's think step-by-step.
1.  **Calculate the total number of ornaments Janet has initially:**
    Janet has 3 boxes, and each box contains 12 ornaments.
    Total ornaments = 3 boxes * 12 ornaments/box = 36 ornaments.

2.  **Calculate the ornaments remaining after decorating the tree:**
    She uses 24 ornaments.
    Ornaments left after decorating = 36 ornaments - 24 ornaments = 12 ornaments.

3.  **Determine the state of the boxes after decorating:**
    Since she used 24 ornaments, and each box has 12, she effectively emptied two boxes (2 * 12 = 24). This means she has one full box of 12 ornaments remaining. The problem statement "her dog knocks over one of the *remaining* boxes" implies there was at least one full box left.

4.  **Calculate the ornaments left after the dog knocks over a box and breaks ornaments:**
    The dog knocks over *one of the remaining boxes*. This implies the 12 ornaments left were in one box.
    This box had 12 ornaments.
    4 ornaments break from this 